# 실습 03 · Cat/Dog pretrained ResNet18 전이학습 · QUIZ

이 노트북은 tutorial을 완료한 후 직접 실습하는 것입니다.
**문제만 주어집니다.** 각 문제를 풀고 정답과 비교하세요.

**시간:** 약 30분  
**평가:** transform 적용, pretrained weight 활용, fine-tuning 이해

**⭐ 푸는 방법:** 각 코드 셀에 뼈대 코드가 주어집니다. `____` 부분만 채운 뒤 실행하세요.
막히면 tutorial의 해당 섹션을 참고해도 됩니다 — 그것도 정식 방법입니다.

## 환경 설정 및 데이터 경로

In [ ]:
import os, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights
from sklearn.model_selection import train_test_split
from pathlib import Path

def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start, *start.parents):
        if (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("data 폴더를 찾지 못했습니다. 노트북과 같은 위치(또는 상위)에 data 폴더가 있어야 합니다.")

PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / "data"
IMAGE_ROOT = DATA_ROOT / "dog_cat"

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

## 문제 1: 이미지 폴더에서 파일 수 확인

다음을 수행하세요:
1. `IMAGE_ROOT / "train" / "cat"` 폴더에서 모든 `.jpg` 파일을 찾으세요.
2. train cat, train dog, test cat, test dog의 각 개수를 출력하세요.
3. 각 폴더의 class label을 정의하세요: `{"cat": 0, "dog": 1}`

**예상:** train ≈ 200개, test ≈ 100개

In [ ]:
# ____ 부분을 채운 뒤 실행하세요
CLASS_TO_IDX = {"cat": 0, "dog": 1}

def collect_paths(split):
    paths, labels = [], []
    for class_name, label in CLASS_TO_IDX.items():
        current = sorted((IMAGE_ROOT / ____ / class_name).glob("*.jpg"))   # train 또는 test 폴더
        paths.extend(current)
        labels.extend([____] * len(current))   # 경로 개수만큼 같은 정답 번호를
    return np.array(paths, dtype=object), np.array(labels, dtype=np.int64)

all_train_paths, all_train_labels = collect_paths("train")
test_paths, test_labels = collect_paths("____")   # 나머지 폴더

print("train:", len(all_train_paths), "| test:", len(test_paths))   # 202 / 102

## 문제 2: 이미지 크기 통계

다음을 수행하세요:
1. 모든 이미지 파일을 돌면서 width, height를 기록하세요.
2. class별로 이미지 크기의 min, median, max를 출력하세요.
3. 이미지들이 다양한 크기라는 것을 확인하세요.

**예상:** 이 수업용 데이터는 전부 32×32로 축소되어 있습니다 (실무 원본은 크기가 제각각)

In [ ]:
# ____ 부분을 채운 뒤 실행하세요
records = []
for path in np.r_[all_train_paths, test_paths]:
    with Image.open(path) as image:
        records.append({"width": image.____, "height": image.____})   # 가로/세로 크기 속성

image_info = pd.DataFrame(records)
display(image_info.agg(["min", "median", "max"]))
# 참고: 이 수업용 데이터는 전부 32×32로 축소되어 있습니다 (실무 원본은 크기가 제각각)

## 문제 3: train/validation/test 분할

다음을 수행하세요:
1. train 폴더의 파일들을 80% train, 20% validation으로 분할하세요.
2. test 폴더는 분할하지 않고 그대로 사용하세요.
3. `stratify=y`를 사용하여 class 균형을 유지하세요.
4. 각 split의 크기와 class 분포를 출력하세요.

**예상:** train ≈ 160, val ≈ 40, test ≈ 100

In [ ]:
# ____ 부분을 채운 뒤 실행하세요
train_paths, val_paths, train_labels, val_labels = train_test_split(
    all_train_paths, all_train_labels, test_size=____, stratify=____, random_state=SEED
)   # 20%를 validation으로, 개·고양이 비율 유지

print("train:", len(train_paths), "| val:", len(val_paths), "| test:", len(test_paths))   # 161 / 41 / 102
print("train class count:", np.bincount(train_labels))

## 문제 4: ResNet18 pretrained weight 확인

다음을 수행하세요:
1. `ResNet18_Weights.DEFAULT`를 로드하세요.
2. weight의 URL을 출력하세요.
3. `WEIGHTS.transforms()`로 권장되는 transform을 확인하세요.
4. ImageNet 평균/표준편차를 추출하세요.

**참고:** pretrained weight는 특정 전처리 규칙을 가정합니다.

## 사전학습 가중치 준비

아래 셀은 교육 자료에 포함된 `weights/resnet18-f37072fd.pth` 파일을 PyTorch 캐시 폴더로 복사합니다.
전원이 동시에 인터넷에서 다운로드하지 않아도 되고, 네트워크 상태와 무관하게 `resnet18(weights=...)`가 바로 동작합니다.
(`weights/` 폴더가 없으면 다음 셀에서 자동으로 인터넷 다운로드를 시도합니다.)

In [ ]:
import shutil
from pathlib import Path
import torch

WEIGHT_NAME = "resnet18-f37072fd.pth"
hub_checkpoints = Path(torch.hub.get_dir()) / "checkpoints"
hub_checkpoints.mkdir(parents=True, exist_ok=True)
cached = hub_checkpoints / WEIGHT_NAME

def find_local_weight(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start, *start.parents):
        weight_path = candidate / "weights" / WEIGHT_NAME
        if weight_path.exists():
            return weight_path
    return None

if cached.exists():
    print("사전학습 가중치가 이미 준비되어 있습니다:", cached)
else:
    local_weight = find_local_weight()
    if local_weight is not None:
        shutil.copy2(local_weight, cached)
        print("가중치 복사 완료:", local_weight, "->", cached)
    else:
        print("경고: weights 폴더를 찾지 못했습니다. 아래 셀에서 인터넷 다운로드를 시도합니다.")

In [ ]:
# ____ 부분을 채운 뒤 실행하세요
WEIGHTS = ResNet18_Weights.____   # 기본(권장) 가중치
print("weight URL:", WEIGHTS.url)
print(WEIGHTS.transforms())

IMAGE_SIZE = ____                 # ResNet18의 입력 크기
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

## 문제 5: train과 eval용 transform 정의

다음을 수행하세요:
1. train_transform을 정의하세요:
   - RandomResizedCrop(224, scale=(0.75, 1.0))
   - RandomHorizontalFlip()
   - ToTensor()
   - Normalize(IMAGENET_MEAN, IMAGENET_STD)
2. eval_transform을 정의하세요:
   - Resize(256)
   - CenterCrop(224)
   - ToTensor()
   - Normalize(IMAGENET_MEAN, IMAGENET_STD)

**설명:** augmentation은 train에만, eval은 결정적입니다.

In [ ]:
# ____ 부분을 채운 뒤 실행하세요
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.____(),                                   # 이미지를 숫자 표(tensor)로
    transforms.Normalize(____, IMAGENET_STD),            # 밝기 기준 (평균, 표준편차)
])
eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(____),                         # 최종 입력 크기
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
print("transform 정의 완료")

## 문제 6: Dataset 클래스 구현

다음 Dataset을 구현하세요:

```python
class CatDogDataset(Dataset):
    def __init__(self, paths, labels, transform):
        # paths: 이미지 파일 경로 list
        # labels: 정수 label list
        # transform: torchvision.transforms.Compose
        pass
    
    def __len__(self):
        pass
    
    def __getitem__(self, index):
        # 이미지 로드 → RGB 변환 → transform 적용 → (image, label) 반환
        pass
```

**반환:** (torch.Tensor, torch.long)

In [ ]:
# ____ 부분을 채운 뒤 실행하세요
class CatDogDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths = list(paths)
        self.labels = list(labels)
        self.transform = transform

    def __len__(self):
        return len(self.____)              # 데이터 개수

    def __getitem__(self, index):
        with Image.open(self.paths[index]) as image:
            image = image.convert("____")   # 색상 형식 통일
        image = self.____(image)            # 가공 적용
        label = torch.tensor(self.labels[index], dtype=torch.____)   # 정답은 정수 타입
        return image, label

print("Dataset 정의 완료")

## 문제 7: DataLoader 구성

다음을 수행하세요:
1. CatDogDataset을 사용하여 train/val/test dataset을 만드세요.
2. DataLoader로 변환하세요 (batch_size=16, train은 shuffle=True).
3. 첫 번째 batch의 shape를 출력하세요.

**예상:** input (16, 3, 224, 224), label (16,)

In [ ]:
# ____ 부분을 채운 뒤 실행하세요
train_set = CatDogDataset(train_paths, train_labels, ____)   # 어느 transform을 짝지을까요?
val_set = CatDogDataset(val_paths, val_labels, eval_transform)
test_set = CatDogDataset(test_paths, test_labels, eval_transform)

train_loader = DataLoader(train_set, batch_size=16, shuffle=____)
val_loader = DataLoader(val_set, batch_size=16, shuffle=False)
test_loader = DataLoader(test_set, batch_size=16, shuffle=False)

images, labels = next(iter(____))                 # 학습용 loader에서 한 batch
print("image batch:", images.shape)               # (16, 3, 224, 224)
print("label batch:", labels.shape, labels.dtype)

## 문제 8: ResNet18 다운로드 및 classifier 교체

다음을 수행하세요:
1. `resnet18(weights=WEIGHTS)`로 pretrained ResNet18을 로드하세요.
2. 모든 파라미터를 `requires_grad=False`로 freeze하세요.
3. `model.fc`를 `nn.Linear(512, 2)`로 교체하세요.
4. 모델을 DEVICE로 옮기세요.
5. frozen vs trainable 파라미터 개수를 출력하세요.

**예상:** frozen ≈ 11M, trainable ≈ 1K

In [ ]:
# ____ 부분을 채운 뒤 실행하세요
model = resnet18(weights=WEIGHTS)

for parameter in model.parameters():
    parameter.requires_grad = ____                       # 전체 고정(freeze)

model.fc = nn.Linear(model.fc.____, ____)                # 기존 층의 입력 크기 / 선택지 수
model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"total: {total_params:,} | trainable: {trainable_params:,}")   # trainable이 1,026이면 성공

## 문제 9: 학습·검증 함수

다음을 수행하세요:
1. `run_epoch(model, loader, criterion, optimizer=None)` 함수를 구현하세요.
2. training 중에만 gradient 계산하세요.
3. (평균 loss, 정확도)를 반환하세요.

**참고:** 이전 quiz와 유사합니다.

In [ ]:
# ____ 부분을 채운 뒤 실행하세요
def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not ____        # optimizer를 받았으면 학습 모드
    model.train(____)                       # 모델에 현재 모드를 알림
    total_loss = total_correct = total_count = 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        if training:
            optimizer.zero_grad()
        with torch.set_grad_enabled(training):
            logits = model(xb)
            loss = criterion(logits, yb)
            if training:
                loss.____()                 # 수정 방향 계산
                optimizer.____()            # 수정 실행
        total_loss += loss.item() * len(yb)
        total_correct += (logits.____(1) == yb).sum().item()   # 점수가 가장 높은 선택지 번호
        total_count += len(yb)
    return total_loss / total_count, total_correct / total_count

print("run_epoch 정의 완료")

## 문제 10: Head-only 학습

다음을 수행하세요:
1. criterion을 CrossEntropyLoss()로 정의하세요.
2. optimizer를 Adam(model.parameters(), lr=1e-3)으로 정의하세요.
3. 5 epoch 동안 학습하세요.
4. 각 epoch마다 train/validation accuracy를 출력하세요.
5. test accuracy를 평가하세요.

**설명:** "head-only"는 classifier만 학습하고 conv 부분은 frozen합니다.

In [ ]:
# ____ 부분을 채운 뒤 실행하세요
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.____(), lr=____)   # 학습할 가중치 목록 / 지정 학습률 (1e-3)

for epoch in range(1, 6):
    tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer)
    va_loss, va_acc = run_epoch(model, val_loader, criterion)
    print(f"epoch {epoch} | train {tr_acc:.3f} | val {va_acc:.3f}")

test_loss, test_acc = run_epoch(model, ____, criterion)   # 최종 평가
print(f"test accuracy: {test_acc:.3f}")